# End-to-End MLP Workflow

This notebook is a fully worked example of how to use `portfolio_toolkit` with a basic MLP-style forecasting model.

It covers the full notebook-first workflow:

1. load a shared dataset
2. add built-in toolkit features
3. engineer new custom notebook-local features
4. build forward return / alpha / volatility targets
5. split into train / validation / test using the shared split rules
6. define and train a small PyTorch MLP regressor
7. emit a standardized prediction table
8. turn predictions into a `PortfolioWeights` object
9. run the shared backtest
10. write reports and artifacts
11. log everything to MLflow

This is intentionally simple and heavily commented. The idea is that a teammate can copy this notebook, replace the model body, keep the shared data/evaluation layer, and still be comparable to everyone else.

## Running This Notebook In Colab

If you want to run this notebook in Google Colab, start by cloning the repo into the Colab session and installing the toolkit in editable mode.

Steps:

1. Set `REPO_URL` below to your GitHub repo URL.
2. Run the bootstrap cell once.
3. After that, the rest of the notebook can import `portfolio_toolkit` normally.

If you are running locally, the same cell will automatically fall back to the repository on your machine.


In [1]:
# Colab / local bootstrap cell
# - In Colab: clone the repo, install the package, and point repo_root at /content/...
# - Locally: just point repo_root at this repository on disk

import sys
from pathlib import Path
 
IN_COLAB = "google.colab" in sys.modules
 
if IN_COLAB:
    REPO_URL = "https://github.com/<your-user-or-org>/<your-repo>.git"  # replace with real URL
    REPO_DIR = "/content/Portfolio-Optimizer"
    if "<your-user-or-org>" in REPO_URL or "<your-repo>" in REPO_URL:
        raise ValueError("Set REPO_URL to your real GitHub repository URL before running in Colab.")
    import subprocess
    subprocess.run(["rm", "-rf", REPO_DIR])
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR])
    import os; os.chdir(REPO_DIR)
    subprocess.run(["pip", "install", "-e", ".[dev]"])
    repo_root = Path(REPO_DIR).resolve()
else:
    repo_root = Path(repo_root).resolve() if "repo_root" in globals() else Path("../../").resolve()
 
print("repo_root =", repo_root)


repo_root = /Users/hannahlee/Documents/MLSN/Portfolio-Optimization-Lib


In [2]:
import sys, pathlib
repo_root = pathlib.Path("/Users/hannahlee/Documents/MLSN/Portfolio-Optimization-Lib")
sys.path.append(str(repo_root / "src"))
from portfolio_toolkit import init_mlflow

/Users/hannahlee/Documents/MLSN/Portfolio-Optimization-Lib/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
#ml flow setup?
from portfolio_toolkit import init_mlflow

mlflow_layout = init_mlflow(repo_root)
print(mlflow_layout)

{'tracking_uri': 'https://adams-macbook-pro.tail5ddc35.ts.net', 'backend_store_uri': 'https://adams-macbook-pro.tail5ddc35.ts.net', 'artifact_root': '', 'db_path': ''}


## What This Example Is Doing

This notebook uses `shared_set_2` by default instead of `shared_set_1`.

Why:

- `shared_set_1` is the full S&P 500 universe, so the first download is much larger.
- `shared_set_2` is smaller and faster for a first MLP example.

Once you understand the pattern, you can switch `dataset_name` to `shared_set_1` and run the exact same workflow over a much broader universe.

In plain English:
“What data am I using, what am I predicting, and where do I save results?”

In [4]:
from pathlib import Path
import numpy as np
import pandas as pd
 
from portfolio_toolkit import (
    build_features,
    build_metrics,
    get_dataset_spec,
    init_mlflow,
    load_prices,
    log_backtest,
    log_portfolio,
    log_predictions,
    make_forward_alpha_target,
    make_forward_realized_vol_target,
    make_forward_return_target,
    slice_split,
    split_dates,
    start_run,
    validate_prediction_frame,
    validate_weights_frame,
    weights_from_predictions_risk_adjusted,
    backtest_weights,
    write_backtest_artifacts,
)

# ---------------------------------------------------------------------
# Basic run configuration.
# ---------------------------------------------------------------------
# repo_root:
#   Where the toolkit config files, cache, MLflow folder, and runs folder live.
# dataset_name:
#   Which shared ticker universe + split rules we want to use.
# horizon:
#   Our prediction horizon in trading days.
# output_dir:
#   Where this notebook will write artifacts like metrics and QuantStats.
# ---------------------------------------------------------------------

repo_root     = Path(repo_root).resolve() if "repo_root" in globals() else Path("../../").resolve()
dataset_name  = "shared_set_2"
model_name    = "autoencoder_mlp_downstream"          # was: torch_mlp_forecast_example
horizon       = 5
output_dir    = repo_root / "runs" / "autoencoder_downstream"  # was: mlp_end_to_end_workflow
output_dir.mkdir(parents=True, exist_ok=True)
 
# FIX: single source of truth for hyperparameters — used in training AND mlflow logging.
AE_EPOCHS    = 25
MLP_EPOCHS   = 30
BATCH_SIZE   = 1024
LATENT_DIM   = 16
LEARNING_RATE = 1e-3
PATIENCE     = 5   # early stopping patience (epochs without val improvement)
 
spec   = get_dataset_spec(dataset_name, repo_root=repo_root)
splits = split_dates(dataset_name, repo_root=repo_root)
 
print("Dataset preset:",       dataset_name)
print("Dataset display name:", spec.name)
print("Tickers modeled:",      len(spec.tickers))
print("Benchmark ticker:",     spec.benchmark_ticker)
print("Train/Val/Test windows:", splits)

Dataset preset: shared_set_2
Dataset display name: growth_tech_innovation
Tickers modeled: 26
Benchmark ticker: SPY
Train/Val/Test windows: {'train': (Timestamp('2014-01-02 00:00:00'), Timestamp('2019-12-31 00:00:00')), 'val': (Timestamp('2020-01-02 00:00:00'), Timestamp('2021-12-31 00:00:00')), 'test': (Timestamp('2022-01-03 00:00:00'), Timestamp('2025-12-31 00:00:00'))}


## 1. Load Shared Price Data

The toolkit's `load_prices(...)` function is the shared data entrypoint.

What it does:

- reads the selected dataset preset from `configs/datasets.toml`
- downloads daily OHLCV data with `yfinance` if it is not already cached
- always includes `SPY` as the benchmark series
- validates and normalizes the dataframe before returning it

This is one of the main standardization points in the repo: everyone starts from the same dataset preset and the same split boundaries.

In plain English:
“Give me clean stock price data everyone on the team uses.”


In [5]:
prices = load_prices(dataset_name, repo_root=repo_root)

print('Price frame shape:', prices.shape)
print('Date range:', prices['date'].min(), '->', prices['date'].max())
print('Number of unique tickers in price frame:', prices['ticker'].nunique())
display(prices.head())

Price frame shape: (78468, 8)
Date range: 2014-01-02 00:00:00 -> 2025-12-31 00:00:00
Number of unique tickers in price frame: 26


,date,ticker,open,high,low,close,adj_close,volume
0,2014-01-02,AAPL,19.845715,19.893929,19.715000,19.754642,17.140659,234684800
1,2014-01-03,AAPL,19.745001,19.775000,19.301071,19.320715,16.764153,392467600
2,2014-01-06,AAPL,19.194643,19.528570,19.057142,19.426071,16.855564,412610800
3,2014-01-07,AAPL,19.440001,19.498571,19.211430,19.287144,16.735022,317209200
4,2014-01-08,AAPL,19.243214,19.484285,19.238930,19.409286,16.841007,258529600


## 2. Add Built-In Toolkit Features

We start with a moderate built-in feature set.

These are all created by the shared library, which means other teammates can use the same starting point if they want:

- momentum
- volatility
- RSI
- moving-average distance
- volume z-score
- benchmark-relative return
- intraday range and beta vs SPY

This is a good example of the intended workflow:

- shared features for consistency and speed
- custom notebook-local features on top when you want to experiment

In plain English:
“Turn raw prices into useful signals the model can learn from.” For example, raw price to momentum, volatility, and RSI


In [6]:
import random #pick random seed for reproducibility
import torch
random.seed(99)
np.random.seed(99)
torch.manual_seed(99)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(99)
torch.backends.cudnn.deterministic = True

In [7]:
base_feature_names = [
    'return_5d',
    'return_20d',
    'vol_20d',
    'momentum_20d',
    'momentum_60d',
    'rsi_14',
    'price_to_sma_20d',
    'price_to_sma_50d',
    'volume_zscore_20d',
    'excess_return_20d_vs_spy',
    'intraday_range',
    'beta_20d_spy',
]

base_features = build_features(prices, feature_names=base_feature_names)
print('Base feature frame shape:', base_features.shape)
display(base_features)

Base feature frame shape: (78468, 14)


,date,ticker,return_5d,return_20d,vol_20d,momentum_20d,momentum_60d,rsi_14,price_to_sma_20d,price_to_sma_50d,volume_zscore_20d,excess_return_20d_vs_spy,intraday_range,beta_20d_spy
0,2014-01-02,AAPL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.009058,NaN
1,2014-01-03,AAPL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.024530,NaN
2,2014-01-06,AAPL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.024268,NaN
3,2014-01-07,AAPL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.014888,NaN
4,2014-01-08,AAPL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.012641,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
78463,2025-12-24,TXN,0.015130,0.094950,0.016532,0.094950,-0.027317,42.843447,0.000438,0.047161,-1.596433,0.069173,0.006888,1.661551
78464,2025-12-26,TXN,0.003916,0.069731,0.016080,0.069731,-0.010706,34.882494,-0.004217,0.045292,-0.964098,0.051090,0.011816,1.574360
78465,2025-12-29,TXN,-0.003403,0.044096,0.015884,0.044096,-0.027764,35.663544,-0.012978,0.038043,-0.696523,0.034595,0.014514,1.532279
78466,2025-12-30,TXN,-0.019014,0.043173,0.015893,0.043173,-0.018491,38.053565,-0.016500,0.036399,-0.727059,0.030281,0.006955,1.581910


## 3. Add New Custom Features In The Notebook

This is where developers keep their freedom.

The toolkit does **not** force everyone to only use built-in features. A normal workflow is:

1. build a shared baseline feature set
2. add experimental features locally in the notebook
3. keep using the shared validation / portfolio / backtest layer afterward

Below we add a few simple handcrafted features:

- `mom_vol_ratio`
  A momentum-to-volatility ratio. This is a quick-and-dirty risk-adjusted trend signal.
- `trend_spread`
  The gap between short-term and medium-term trend distance.
- `quality_signal`
  A simple benchmark-relative momentum signal penalized by volatility.
- `range_volume_interaction`
  A rough interaction term between price range expansion and unusual volume.

In plain English:
“Add your own ideas on top of the default signals.”


In [8]:
frame = base_features.copy()

# Add a very small constant anywhere we divide so we do not create infinities.
eps = 1e-6

frame['mom_vol_ratio'] = frame['momentum_20d'] / (frame['vol_20d'].abs() + eps)
frame['trend_spread'] = frame['price_to_sma_20d'] - frame['price_to_sma_50d']
frame['quality_signal'] = frame['excess_return_20d_vs_spy'] - 0.5 * frame['vol_20d']
frame['range_volume_interaction'] = frame['intraday_range'] * frame['volume_zscore_20d']

custom_feature_names = [
    'mom_vol_ratio',
    'trend_spread',
    'quality_signal',
    'range_volume_interaction',
]

all_feature_names = base_feature_names + custom_feature_names
display(frame.loc[:, ['date', 'ticker'] + custom_feature_names].head())

,date,ticker,mom_vol_ratio,trend_spread,quality_signal,range_volume_interaction
0,2014-01-02,AAPL,NaN,NaN,NaN,NaN
1,2014-01-03,AAPL,NaN,NaN,NaN,NaN
2,2014-01-06,AAPL,NaN,NaN,NaN,NaN
3,2014-01-07,AAPL,NaN,NaN,NaN,NaN
4,2014-01-08,AAPL,NaN,NaN,NaN,NaN


## 4. Build Targets

We are going to fit three separate small MLPs with the exact same input features:

- one model for `expected_return`
- one model for `expected_alpha`
- one model for `expected_volatility`

This is not required, but it demonstrates the richer prediction contract supported by the toolkit.

Target builders used here:

- `make_forward_return_target(...)`
- `make_forward_alpha_target(...)`
- `make_forward_realized_vol_target(...)`

In plain English:
“What should the model learn to predict?”


In [9]:
return_targets = make_forward_return_target(prices, horizon=horizon)
alpha_targets = make_forward_alpha_target(prices, horizon=horizon)
vol_targets = make_forward_realized_vol_target(prices, window=horizon)

target_frame = frame.merge(return_targets, on=['date', 'ticker'], how='left')
target_frame = target_frame.merge(alpha_targets, on=['date', 'ticker'], how='left')
target_frame = target_frame.merge(vol_targets, on=['date', 'ticker'], how='left')

# Drop rows only after all features and targets are assembled.
# This is the usual notebook pattern because long-window features and forward targets
# naturally create missing values near the beginning and end of each ticker history.
target_frame = target_frame.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

return_target_col = f'forward_return_{horizon}d'
alpha_target_col = f'forward_alpha_{horizon}d_vs_spy'
vol_target_col = f'forward_realized_vol_{horizon}d'

print('Modeling frame shape after dropping nulls:', target_frame.shape)
display(target_frame.head())

Modeling frame shape after dropping nulls: (76765, 21)


,date,ticker,return_5d,return_20d,vol_20d,momentum_20d,momentum_60d,rsi_14,price_to_sma_20d,price_to_sma_50d,...,excess_return_20d_vs_spy,intraday_range,beta_20d_spy,mom_vol_ratio,trend_spread,quality_signal,range_volume_interaction,forward_return_5d,forward_alpha_5d_vs_spy,forward_realized_vol_5d
0,2014-03-31,AAPL,-0.004544,0.017015,0.006837,0.017015,-0.023823,50.700672,0.006098,0.014112,...,0.001579,0.009092,0.415702,2.488169,-0.008015,-0.001840,-0.011534,-0.024723,-0.010445,0.146485
1,2014-04-01,AAPL,-0.006129,0.019596,0.006966,0.019596,0.007232,54.962914,0.014312,0.023227,...,0.011594,0.009416,0.468962,2.812699,-0.008915,0.008111,-0.005887,-0.033620,-0.016886,0.108589
2,2014-04-02,AAPL,0.005132,0.019142,0.006963,0.019142,0.003434,63.014296,0.015029,0.025053,...,0.008683,0.005935,0.465292,2.748665,-0.010024,0.005202,-0.005713,-0.022542,-0.013066,0.163987
3,2014-04-03,AAPL,0.002475,0.015148,0.007125,0.015148,0.003657,66.198799,0.007236,0.018312,...,0.008333,0.009020,0.501709,2.125658,-0.011076,0.004770,-0.011168,-0.028415,0.000583,0.172602
4,2014-04-04,AAPL,-0.009388,0.002601,0.007727,0.002601,-0.015561,55.243215,-0.005922,0.005939,...,0.008111,0.017713,0.631212,0.336625,-0.011861,0.004248,0.012090,-0.022959,0.003276,0.164259


## 5. Shared Train / Validation / Test Splits And Feature Scaling

This section does two very important things:

1. Uses the repo's shared split functions so the notebook respects the official date windows.
2. Standardizes features **using only the training split statistics**.

That second part matters a lot.

We do **not** want to normalize using future information from validation or test rows. So we compute mean and standard deviation from the train split only, then reuse those values everywhere else.

In plain English:
“Learn on past data, test on future data without cheating.”


In [10]:
train = slice_split(target_frame, dataset_name, "train", repo_root=repo_root)
val   = slice_split(target_frame, dataset_name, "val",   repo_root=repo_root)
test  = slice_split(target_frame, dataset_name, "test",  repo_root=repo_root)
 
print("Train rows:", len(train))
print("Val rows:",   len(val))
print("Test rows:",  len(test))
 
# Standardize using training-set statistics only.
train_means = train[all_feature_names].mean()
train_stds  = train[all_feature_names].std(ddof=0).replace(0.0, 1.0)
 
def standardize(feature_frame: pd.DataFrame) -> np.ndarray:
    return ((feature_frame[all_feature_names] - train_means) / train_stds).to_numpy(dtype=float)
 
X_train = standardize(train)
X_val   = standardize(val)
X_test  = standardize(test)
 
y_train_return = train[return_target_col].to_numpy(dtype=float)
y_val_return   = val[return_target_col].to_numpy(dtype=float)
y_test_return  = test[return_target_col].to_numpy(dtype=float)
 
y_train_alpha  = train[alpha_target_col].to_numpy(dtype=float)
y_val_alpha    = val[alpha_target_col].to_numpy(dtype=float)
 
y_train_vol    = train[vol_target_col].to_numpy(dtype=float)
y_val_vol      = val[vol_target_col].to_numpy(dtype=float)
 
print("X_train shape:", X_train.shape)
print("Feature count:", X_train.shape[1])

Train rows: 37695
Val rows: 13130
Test rows: 25940
X_train shape: (37695, 16)
Feature count: 16


## 6. Define A Very Basic MLP Class

This is a tiny PyTorch implementation of a feed-forward neural network for regression.

It is intentionally simple so the workflow is easy to understand:

- fully connected layers
- ReLU hidden activations
- linear output layer
- mean squared error loss
- mini-batch gradient descent
- validation loss tracking

This is **not** meant to be the most advanced PyTorch training setup. It is here so the notebook shows a real neural-network workflow while still staying readable.

In a real team workflow, this cell is exactly the part a researcher would replace with their own model implementation.

In plain English:
“This is the brain that learns patterns.”


In [11]:
#Define models
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
 
 
class Autoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim=LATENT_DIM):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.Linear(64, input_dim),
        )
 
    def forward(self, x):
        return self.decoder(self.encoder(x))
 
 
class TorchMLPRegressor(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )
 
    def forward(self, x):
        return self.net(x)

Using device: cpu


## 7. Train Three Small MLPs

We are using the same feature matrix to train three related regressors:

- return model
- alpha model
- volatility model

This is a very common pattern in research projects:

- one common feature pipeline
- multiple prediction heads or target-specific models

The code below wraps the repeated steps in a helper function so the notebook stays readable.

In plain English:
“Teach the model what patterns lead to future returns.”


In [12]:
#train helpers
def train_autoencoder(X_tr, X_va):
    """Trains the autoencoder with minibatching and early stopping."""
    model     = Autoencoder(input_dim=X_tr.shape[1]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    loss_fn   = nn.MSELoss()
 
    X_tr_t = torch.tensor(X_tr, dtype=torch.float32).to(device)
    X_va_t = torch.tensor(X_va, dtype=torch.float32).to(device)
 
    loader = DataLoader(
        TensorDataset(X_tr_t, X_tr_t),
        batch_size=BATCH_SIZE,
        shuffle=True,
    )
 
    best_val_loss = float("inf")
    patience_counter = 0
 
    for epoch in range(AE_EPOCHS):
        model.train()
        epoch_loss = 0.0
        for xb, _ in loader:
            optimizer.zero_grad()
            loss = loss_fn(model(xb), xb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(xb)
        epoch_loss /= len(X_tr)
 
        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(X_va_t), X_va_t).item()
 
        print(f"AE  Epoch {epoch+1:3d}/{AE_EPOCHS} | train loss: {epoch_loss:.6f} | val loss: {val_loss:.6f}")
 
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"  Early stopping at epoch {epoch+1} (no val improvement for {PATIENCE} epochs).")
                break
 
    model.load_state_dict(best_state)
    return model
 
 
def train_predictor(X_tr, y_tr, X_va, y_va, label="MLP"):
    """Trains a downstream MLP predictor with minibatching and early stopping."""
    model     = TorchMLPRegressor(input_dim=X_tr.shape[1]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    loss_fn   = nn.MSELoss()
 
    X_tr_t = torch.tensor(X_tr, dtype=torch.float32).to(device)
    y_tr_t = torch.tensor(y_tr.reshape(-1, 1), dtype=torch.float32).to(device)
    X_va_t = torch.tensor(X_va, dtype=torch.float32).to(device)
    y_va_np = y_va.reshape(-1)
 
    loader = DataLoader(
        TensorDataset(X_tr_t, y_tr_t),
        batch_size=BATCH_SIZE,
        shuffle=True,
    )
 
    best_val_loss = float("inf")
    patience_counter = 0
 
    for epoch in range(MLP_EPOCHS):
        model.train()
        for xb, yb in loader:
            optimizer.zero_grad()
            loss_fn(model(xb), yb).backward()
            optimizer.step()
 
        model.eval()
        with torch.no_grad():
            val_pred = model(X_va_t).cpu().numpy().reshape(-1)
        val_loss = np.mean((val_pred - y_va_np) ** 2)
 
        print(f"{label} Epoch {epoch+1:3d}/{MLP_EPOCHS} | val MSE: {val_loss:.6f}")
 
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"  Early stopping at epoch {epoch+1}.")
                break
 
    model.load_state_dict(best_state)
    return model
 
 
def encode(ae_model, X):
    """Extract latent representations from a trained autoencoder encoder."""
    ae_model.eval()
    with torch.no_grad():
        return ae_model.encoder(torch.tensor(X, dtype=torch.float32).to(device)).cpu().numpy()
 
 
def predict(model, X):
    """Run inference with a trained predictor."""
    model.eval()
    with torch.no_grad():
        return model(torch.tensor(X, dtype=torch.float32).to(device)).cpu().numpy().reshape(-1)
    
#train autoencoder
autoencoder = train_autoencoder(X_train, X_val)
 
Z_train = encode(autoencoder, X_train)
Z_val   = encode(autoencoder, X_val)
Z_test  = encode(autoencoder, X_test)
 


#train downstream predictors
print("--- Training return predictor ---")
return_model = train_predictor(Z_train, y_train_return, Z_val, y_val_return, label="Return MLP")
 
print("\n--- Training alpha predictor ---")
alpha_model  = train_predictor(Z_train, y_train_alpha,  Z_val, y_val_alpha,  label="Alpha  MLP")

AE  Epoch   1/25 | train loss: 0.774875 | val loss: 0.665345
AE  Epoch   2/25 | train loss: 0.332426 | val loss: 0.329312
AE  Epoch   3/25 | train loss: 0.166747 | val loss: 0.196081
AE  Epoch   4/25 | train loss: 0.102159 | val loss: 0.129663
AE  Epoch   5/25 | train loss: 0.070585 | val loss: 0.109611
AE  Epoch   6/25 | train loss: 0.058000 | val loss: 0.094477
AE  Epoch   7/25 | train loss: 0.048291 | val loss: 0.079161
AE  Epoch   8/25 | train loss: 0.039451 | val loss: 0.066152
AE  Epoch   9/25 | train loss: 0.032102 | val loss: 0.055404
AE  Epoch  10/25 | train loss: 0.025522 | val loss: 0.046104
AE  Epoch  11/25 | train loss: 0.019680 | val loss: 0.037858
AE  Epoch  12/25 | train loss: 0.015456 | val loss: 0.030514
AE  Epoch  13/25 | train loss: 0.012596 | val loss: 0.024475
AE  Epoch  14/25 | train loss: 0.010820 | val loss: 0.019656
AE  Epoch  15/25 | train loss: 0.009682 | val loss: 0.017516
AE  Epoch  16/25 | train loss: 0.008851 | val loss: 0.015762
AE  Epoch  17/25 | train

## 8. Create The Standardized Prediction Table

Now we convert raw model outputs into the shared prediction contract used by the rest of the toolkit.

Required columns:

- `date`
- `ticker`
- `horizon`
- `expected_return`

Optional columns we will also populate:

- `expected_alpha`
- `expected_volatility`
- `uncertainty`

For this notebook, `uncertainty` is just a simple constant based on validation RMSE for the return model. That is not a sophisticated uncertainty estimate. It is only here to demonstrate where that information would live in the shared schema.

In plain English:
“For each stock, what do we think will happen next?”


In [13]:
# HOW IT WORKS:
#   1. Compute each ticker's daily log return from the shared price frame.
#   2. Roll a 20-day std over those returns.
#   3. Annualize by multiplying by sqrt(252).
#   4. Join onto the test set by (date, ticker).
#   5. Fall back to the cross-sectional mean for any missing values.
# =============================================================================
 
# --- Step 1-3: compute rolling realized vol per ticker ---
prices_sorted = prices.sort_values(["ticker", "date"]).copy()
prices_sorted["log_return"] = (
    prices_sorted
    .groupby("ticker")["close"]          # use whichever price column your toolkit provides
    .transform(lambda s: np.log(s).diff())
)
prices_sorted["realized_vol_20d"] = (
    prices_sorted
    .groupby("ticker")["log_return"]
    .transform(lambda s: s.rolling(20, min_periods=10).std() * np.sqrt(252))
)
 
vol_lookup = prices_sorted[["date", "ticker", "realized_vol_20d"]].dropna()
 
# --- Step 4: join onto test rows ---
test_with_vol = test[["date", "ticker"]].merge(vol_lookup, on=["date", "ticker"], how="left")
 
# --- Step 5: fill any missing with cross-sectional mean ---
fallback_vol = vol_lookup["realized_vol_20d"].mean()
realized_vol_test = test_with_vol["realized_vol_20d"].fillna(fallback_vol).to_numpy()
 
print(f"Realized vol stats (test set):")
print(f"  mean: {realized_vol_test.mean():.4f}")
print(f"  std:  {realized_vol_test.std():.4f}")
print(f"  min:  {realized_vol_test.min():.4f}")
print(f"  max:  {realized_vol_test.max():.4f}")
print(f"  missing filled with fallback: {test_with_vol['realized_vol_20d'].isna().sum()} rows")
 
# Sanity check — vol and |return| should now be on similar scales
test_return_pred = predict(return_model, Z_test)
print(f"\nScale check:")
print(f"  |expected_return| mean: {np.abs(test_return_pred).mean():.4f}")
print(f"  realized_vol mean:      {realized_vol_test.mean():.4f}")
print(f"  ratio (vol / |ret|):    {realized_vol_test.mean() / np.abs(test_return_pred).mean():.1f}x  (target: < 20x)")
 
# --- Build prediction frame ---
test_alpha_pred = predict(alpha_model, Z_test)
 
predictions = test.loc[:, ["date", "ticker"]].copy()
predictions["horizon"]             = horizon
predictions["expected_return"]     = test_return_pred
predictions["expected_alpha"]      = test_alpha_pred
predictions["expected_volatility"] = np.clip(realized_vol_test, 1e-4, None)
predictions["uncertainty"]         = np.abs(test_return_pred - test_return_pred.mean())
 
predictions = validate_prediction_frame(
    predictions,
    dataset_name=dataset_name,
    horizon=horizon,
    repo_root=repo_root,
)
display(predictions.head())
 

Realized vol stats (test set):
  mean: 0.3766
  std:  0.1752
  min:  0.0566
  max:  1.6649
  missing filled with fallback: 0 rows

Scale check:
  |expected_return| mean: 0.0123
  realized_vol mean:      0.3766
  ratio (vol / |ret|):    30.7x  (target: < 20x)


,date,ticker,horizon,expected_return,expected_alpha,expected_volatility,uncertainty
0,2022-01-03,AAPL,5,0.011513,0.004206,0.306400,0.000146
1,2022-01-03,ADBE,5,0.019185,0.028133,0.540421,0.007525
2,2022-01-03,ADI,5,0.006414,0.024410,0.261626,0.005246
3,2022-01-03,AMAT,5,0.011576,0.011920,0.448467,0.000083
4,2022-01-03,AMD,5,0.003294,0.014600,0.591753,0.008366


## 9. Turn Predictions Into A Portfolio Object

The toolkit separates forecasting from portfolio construction.

Here we use the built-in `weights_from_predictions_risk_adjusted(...)` helper.

What it does:

- uses `expected_return / expected_volatility` as the score
- keeps the allocation long-only
- normalizes the scores so each row sums to `1.0`
- returns a `PortfolioWeights` object

This is a good default for demonstrations because it uses more of the prediction contract than a plain top-k rule.

In plain English:
“Use predictions to decide how to allocate money.”


In [14]:
portfolio = weights_from_predictions_risk_adjusted(
    predictions,
    dataset_name=dataset_name,
    strategy_name=model_name,
)
validated_weights = validate_weights_frame(
    portfolio.weights,
    dataset_name=dataset_name,
    repo_root=repo_root,
)
print("Strategy name:",        portfolio.strategy_name)
print("Weights frame shape:",  validated_weights.shape)
display(validated_weights.head())

Strategy name: autoencoder_mlp_downstream
Weights frame shape: (998, 26)


,AAPL,ADBE,ADI,AMAT,AMD,AMZN,AVGO,CRM,CSCO,GOOGL,...,MU,NFLX,NOW,NVDA,ORCL,PANW,QCOM,SPY,TSLA,TXN
date,,,,,,,,,,,,,,,,,,,,,
2022-01-03,0.047252,0.044642,0.030828,0.032460,0.007000,0.015113,0.029924,0.031371,0.049088,0.048868,...,0.013510,0.038184,0.061214,0.047799,0.031902,0.041194,0.057685,0.070802,0.043290,0.017911
2022-01-04,0.037969,0.041166,0.015932,0.021349,0.032641,0.020454,0.033680,0.042787,0.060593,0.047417,...,0.014775,0.030193,0.083655,0.034841,0.027417,0.059404,0.045809,0.052704,0.033201,0.023684
2022-01-05,0.025957,0.039672,0.025436,0.039856,0.059774,0.027039,0.032174,0.013719,0.009716,0.038148,...,0.002130,0.059384,0.061134,0.026772,0.032561,0.074837,0.044984,0.055982,0.019768,0.028472
2022-01-06,0.061075,0.053127,0.010006,0.031135,0.061343,0.057319,0.051751,0.062158,0.031123,0.040163,...,0.003260,0.024399,0.071621,0.023331,0.035060,0.021521,0.027274,0.048170,0.036547,0.020510
2022-01-07,0.047098,0.038546,0.028770,0.048659,0.053046,0.054656,0.053731,0.049266,0.019347,0.039160,...,0.015682,0.066802,0.048276,0.027274,0.023046,0.025597,0.036602,0.038516,0.038554,0.015957


## 10. Run The Shared Backtest

This is where the toolkit gives you the most value as shared infrastructure.

The backtest layer will:

- load the shared dataset prices
- align rebalance dates to the next available trading day
- apply transaction costs from the dataset preset
- compare the strategy to benchmarks like `SPY`
- compute NAV, returns, turnover, and summary metrics

Because this is shared across the team, different notebooks remain comparable even if the model logic is very different.

In plain English:
“If we followed this strategy in the past, how much money would we make?”


In [15]:
result       = backtest_weights(dataset_name, portfolio, repo_root=repo_root)
metrics      = build_metrics(result)
artifact_paths = write_backtest_artifacts(result, output_dir)
 
metrics_table = (
    pd.DataFrame([{"metric": k, "value": v} for k, v in sorted(metrics.items())])
    .sort_values("metric")
    .reset_index(drop=True)
)
display(metrics_table)
print("QuantStats report:", artifact_paths["quantstats_report"])

,metric,value
0,annual_return,0.011606
1,annual_volatility,0.163015
2,average_turnover,0.205685
3,benchmark_total_return,0.507582
4,calmar,0.026063
5,excess_return_vs_benchmark,-0.359336
6,max_drawdown,-0.445295
7,sharpe,0.071193
8,sortino,0.058966
9,total_return,0.148246


QuantStats report: /Users/hannahlee/Documents/MLSN/Portfolio-Optimization-Lib/runs/autoencoder_downstream/quantstats.html


## 11. Log The Run To MLflow

The toolkit keeps MLflow intentionally lightweight:

- local SQLite backend
- local artifact storage
- notebook-friendly logging helpers

The pattern here is the one you want teammates to reuse:

1. initialize MLflow locally
2. start a run with meaningful tags
3. log predictions, portfolio weights, and backtest results
4. let MLflow keep the artifact trail

In plain English:
“Record everything so we can compare runs later.”


In [16]:
mlflow_layout = init_mlflow(repo_root)
print("MLflow tracking URI:", mlflow_layout["tracking_uri"])
 
with start_run(
    run_name="Hannah_Init",
    dataset_name=dataset_name,
    tags={
        "workflow":           "autoencoder_downstream",
        "model_family":       "autoencoder",
        "prediction_horizon": str(horizon),
    },
    repo_root=repo_root,
):
    import mlflow
 
    mlflow.log_params({
        "model_name":          model_name,
        "dataset_name":        dataset_name,
        "horizon":             horizon,
        "feature_count":       len(all_feature_names),
        "base_feature_list":   ",".join(base_feature_names),
        "custom_feature_list": ",".join(custom_feature_names),
        "latent_dim":          LATENT_DIM,
        "ae_epochs":           AE_EPOCHS,        # was: 'epochs': 35 (wrong)
        "mlp_epochs":          MLP_EPOCHS,        # was: missing
        "batch_size":          BATCH_SIZE,
        "learning_rate":       LEARNING_RATE,
        "early_stopping_patience": PATIENCE,
        "ae_hidden_dims":      "64",
        "mlp_hidden_dims":     "32",
        "portfolio_builder":   "weights_from_predictions_risk_adjusted",
        'random_seed': 99,
        "cost_bps":            spec.cost_bps,
    })
 
    log_predictions(predictions)
    log_portfolio(portfolio)
    log_backtest(result)
    print("MLflow logging complete.")

MLflow tracking URI: https://adams-macbook-pro.tail5ddc35.ts.net
MLflow logging complete.
🏃 View run Hannah_Init at: https://adams-macbook-pro.tail5ddc35.ts.net/#/experiments/1/runs/2a72f0f04c7b44d9a7f21d0a001f1dc2
🧪 View experiment at: https://adams-macbook-pro.tail5ddc35.ts.net/#/experiments/1


## 12. Inspect Results

At this point the notebook has produced:

- validated predictions
- validated portfolio weights
- a shared backtest result
- standardized performance metrics
- a QuantStats HTML report
- an MLflow run with artifacts and metrics

That is the full intended research loop for this repo.

In plain English:
“Did this strategy actually work?”


In [17]:
print("Top-level metrics:")
for key, value in sorted(result.metrics.items()):
    print(f"  {key}: {value:.6f}")
 
print("\nArtifact paths:")
for key, value in artifact_paths.items():
    print(f"  {key}: {value}")
 
display(result.nav.tail().to_frame("nav"))
display(result.returns.tail().to_frame("returns"))
display(result.turnover.tail().to_frame("turnover"))

Top-level metrics:
  annual_return: 0.011606
  annual_volatility: 0.163015
  average_turnover: 0.205685
  benchmark_total_return: 0.507582
  calmar: 0.026063
  excess_return_vs_benchmark: -0.359336
  max_drawdown: -0.445295
  sharpe: 0.071193
  sortino: 0.058966
  total_return: 0.148246

Artifact paths:
  weights: /Users/hannahlee/Documents/MLSN/Portfolio-Optimization-Lib/runs/autoencoder_downstream/weights.parquet
  nav: /Users/hannahlee/Documents/MLSN/Portfolio-Optimization-Lib/runs/autoencoder_downstream/nav.parquet
  returns: /Users/hannahlee/Documents/MLSN/Portfolio-Optimization-Lib/runs/autoencoder_downstream/returns.parquet
  turnover: /Users/hannahlee/Documents/MLSN/Portfolio-Optimization-Lib/runs/autoencoder_downstream/turnover.parquet
  benchmarks: /Users/hannahlee/Documents/MLSN/Portfolio-Optimization-Lib/runs/autoencoder_downstream/benchmarks.parquet
  metrics: /Users/hannahlee/Documents/MLSN/Portfolio-Optimization-Lib/runs/autoencoder_downstream/metrics.json
  metrics_tabl

,nav
2025-12-24,116.744326
2025-12-26,116.721099
2025-12-29,116.319297
2025-12-30,116.041283
2025-12-31,114.824604


,returns
2025-12-24,0.004091
2025-12-26,-0.000199
2025-12-29,-0.003442
2025-12-30,-0.002390
2025-12-31,-0.010485


,turnover
date,
2025-12-17,0.212027
2025-12-18,0.151730
2025-12-19,0.341406
2025-12-22,0.253784
2025-12-23,0.126805


In [18]:
# ---------------------------------------------------------------------
# Final validation cell.
# ---------------------------------------------------------------------
# These checks are intentionally simple. They are the kind of sanity checks
# you want at the end of a notebook before you trust the output.

# In plain English:
# “Make sure nothing is broken before calling it done.”

# ---------------------------------------------------------------------

assert {"total_return", "annual_return", "sharpe", "max_drawdown"}.issubset(result.metrics)
assert validated_weights.index.is_monotonic_increasing
assert (validated_weights.sum(axis=1).round(6) == 1.0).all()
assert Path(artifact_paths["quantstats_report"]).exists()
assert {"date", "ticker", "horizon", "expected_return"}.issubset(predictions.columns)
assert predictions["uncertainty"].nunique() > 1, "uncertainty must be per-asset, not a scalar"
assert predictions["expected_alpha"].nunique() > 1, "expected_alpha must differ from expected_return"
print("End-to-end autoencoder workflow validated successfully.")

End-to-end autoencoder workflow validated successfully.
